In [0]:
import mlflow
import time
import json
from pyspark.sql import functions as 
import sys
import os
from pathlib import Path


In [0]:

import sys
import os
from pathlib import Path

REPO_ROOT = str(Path(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()).parent.parent)

from utils.config import *
from utils.states import *
from utils.llm_utils import *

LLM_API_KEY = get_api_key(dbutils)

In [0]:
df_extracted = spark.table(LLM_EXTRACTED).filter(
    F.col("llm_status") == "success"
)

# Confirm product_description column exists
display(df_extracted.select(
    "product_id",
    "product_description",
    "extracted_name",
    "extracted_brand",
    "extracted_sub_category"
))

In [0]:
def judge_product(
    product_description: str,
    name               : str,
    brand              : str,
    sub_category       : str
) -> dict:
    """
    TechMart-specific judge logic.
    Receives description + extracted fields, independently classifies
    the product and verifies if extractor sub_category is correct.
    """
    system_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_JUDGE, role="system",
        approved_taxonomy=APPROVED_TAXONOMY
    )
    user_prompt = load_prompt(
        PROMPTS_DIR, PROMPT_JUDGE, role="user",
        product_description=product_description,
        name=name,
        brand=brand,
        sub_category=sub_category
    )

    result = call_llm(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        api_key      = LLM_API_KEY,
        api_url      = LLM_API_URL,
        model        = LLM_MODEL,
        max_tokens   = 100,
        output_model = JudgeResult
    )

    judge_result = result["validated_output"]

    return {
        "judge_taxonomy" : judge_result.judge_taxonomy,
        "judge_approved" : judge_result.judge_approved,
        "reason"         : judge_result.reason,
        "input_tokens"   : result["input_tokens"],
        "output_tokens"  : result["output_tokens"],
        "latency"        : result["latency_seconds"]
    }

In [0]:

mlflow.set_experiment(MLFLOW_EXPERIMENT)

records     = df_extracted.collect()
all_records = []

with mlflow.start_run(run_name=f"judge_{PROMPT_VERSION}"):

    # Log judge prompt template as artifact
    judge_template_raw = load_prompt_template_raw(PROMPTS_DIR, PROMPT_JUDGE)
    mlflow.log_text(judge_template_raw, PROMPT_JUDGE)

    # Log run parameters
    mlflow.log_param("prompt_version", PROMPT_VERSION)
    mlflow.log_param("model",          LLM_MODEL)
    mlflow.log_param("provider",       LLM_PROVIDER)
    mlflow.log_param("total_records",  len(records))

    approved_count   = 0
    flagged_count    = 0
    total_input_tok  = 0
    total_output_tok = 0
    total_latency    = 0.0

    for row in records:
        product_id          = row["product_id"]
        product_description = row["product_description"]  # ← add this
        name                = row["extracted_name"]
        brand               = row["extracted_brand"]
        sub_category        = row["extracted_sub_category"]

        try:
            result = judge_product(
                product_description=product_description,  # ← pass it here
                name=name,
                brand=brand,
                sub_category=sub_category
            )

            total_input_tok  += result["input_tokens"]
            total_output_tok += result["output_tokens"]
            total_latency    += result["latency"]

            if result["judge_approved"]:    # ← fixed key
                approved_count += 1
            else:
                flagged_count += 1

            all_records.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_taxonomy"        : result["judge_taxonomy"],   # ← add this
                "judge_approved"        : str(result["judge_approved"]),  # ← fixed key
                "judge_reason"          : result["reason"],
                "prompt_version"        : PROMPT_VERSION
            })

        except Exception as e:
            flagged_count += 1
            all_records.append({
                "product_id"            : str(product_id),
                "product_description"   : product_description,
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_taxonomy"        : None,
                "judge_approved"        : "False",
                "judge_reason"          : f"Judge call failed: {str(e)}",
                "prompt_version"        : PROMPT_VERSION
            })

        time.sleep(0.3)
        
    # Log all metrics to MLflow
    mlflow.log_metric("approved_count",      approved_count)
    mlflow.log_metric("flagged_count",       flagged_count)
    mlflow.log_metric("approval_rate",       round(approved_count / len(records), 3))
    mlflow.log_metric("total_input_tokens",  total_input_tok)
    mlflow.log_metric("total_output_tokens", total_output_tok)
    mlflow.log_metric("total_tokens",        total_input_tok + total_output_tok)
    mlflow.log_metric("avg_latency_seconds", round(total_latency / len(records), 3))

print(f"✅ Judging complete: {approved_count} approved, {flagged_count} flagged")

In [0]:
df_taxonomy = spark.createDataFrame(all_records)

(df_taxonomy.select(
 'product_id',
 'product_description',
 'extracted_brand',
 'extracted_name',
 'extracted_sub_category',
 'judge_taxonomy',
 'judge_approved',
 'judge_reason',
 'prompt_version'
 ).write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    # .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TAXONOMY))

print(f"✅ Written: {SILVER_TAXONOMY}")
display(df_taxonomy)

In [0]:
print("=== VALIDATION ===")
print(f"Total records : {spark.table(SILVER_TAXONOMY).count()}")
print(f"Approved      : {spark.table(SILVER_TAXONOMY).filter(F.col('judge_approved') == 'True').count()}")
print(f"Flagged       : {spark.table(SILVER_TAXONOMY).filter(F.col('judge_approved') == 'False').count()}")

display(spark.table(SILVER_TAXONOMY))